# Text Classification — Machine Learning Assignment (Task 2)

HCMUT · Semester I, 2026–2027. **Runtime → Run all** reproduces every number in the report (about 10–15 min on a T4 GPU).

Three pipelines are compared on three public corpora:

| Pipeline | Features | Classifier |
| --- | --- | --- |
| Traditional | TF-IDF (saved as `.h5`) | Naive Bayes / Logistic Regression / Linear SVM |
| Modern features | frozen BGE-small sentence embeddings (saved as `.npy`) | Logistic Regression |
| Deep learning (bonus) | embeddings learned end-to-end | TextCNN |

Data are downloaded from Hugging Face: [AG News](https://huggingface.co/datasets/fancyzhx/ag_news),
[Rotten Tomatoes](https://huggingface.co/datasets/cornell-movie-review-data/rotten_tomatoes),
[Banking77](https://huggingface.co/datasets/mteb/banking77); encoder [BGE-small](https://huggingface.co/BAAI/bge-small-en-v1.5).
No Google Drive is mounted.

## 1. Install libraries
Colab already ships NumPy, pandas, scikit-learn, PyTorch and matplotlib; this adds the rest.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets>=2.19", "transformers>=4.40", "h5py>=3.10"])

## 2. Write the project modules
The cell below is generated from `modules/` in the repository (`notebooks/build_notebook.py`); do not edit it by hand.

In [ ]:
from pathlib import Path
SOURCES = {
 "__init__.py": "\"\"\"Text classification pipeline for the HCMUT Machine Learning assignment (Task 2).\"\"\"\nfrom .config import DATASETS\nfrom .pipeline import FAMILIES, run_dataset\n\n__all__ = [\"DATASETS\", \"FAMILIES\", \"run_dataset\"]\n",
 "config.py": "\"\"\"All experiment settings in one place.\"\"\"\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parents[1]\nSEED = 42\nVAL_FRACTION = 0.15  # stratified share of the official training split used for validation\n\n# Each corpus: Hugging Face id, optional stratified caps on train/test, and the\n# token budget used by BGE and TextCNN (chosen above the 95th-percentile length).\nDATASETS = {\n    \"ag_news\": dict(hf_id=\"fancyzhx/ag_news\", train_cap=10_000, test_cap=2_000, max_len=96),\n    \"rotten_tomatoes\": dict(hf_id=\"cornell-movie-review-data/rotten_tomatoes\", train_cap=None, test_cap=None, max_len=64),\n    \"banking77\": dict(hf_id=\"mteb/banking77\", train_cap=None, test_cap=None, max_len=64),\n}\nQUICK_CAPS = (1_500, 600)  # train/test caps for the --quick smoke test\n\n# Traditional pipeline: every preprocessing combination x every classifier setting.\nTFIDF_GRID = dict(tokenizer=(\"regex\", \"whitespace\"), stop_words=(False, True), max_ngram=(1, 2))\nTFIDF_PARAMS = dict(min_df=2, max_features=25_000, sublinear_tf=True)\nCLASSIFIER_GRID = {\n    \"nb\": (0.01, 0.03, 0.1, 0.3, 1, 3),  # Multinomial Naive Bayes smoothing alpha\n    \"logreg\": (1, 4, 16, 64),      # inverse regularisation strength C\n    \"svm\": (0.1, 0.3, 1, 3),       # LinearSVC C\n}\n\n# Modern pipeline: frozen sentence encoder + logistic regression.\nBGE_MODEL = \"BAAI/bge-small-en-v1.5\"\nBGE_C_GRID = (0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0)\n\n# Deep-learning pipeline (bonus): TextCNN trained end-to-end.\nTEXTCNN = dict(vocab_size=20_000, min_count=2, emb_dim=100, n_filters=64, widths=(3, 4, 5),\n               dropout=0.5, lr=1e-3, max_epochs=20, patience=3)\n\nN_BOOTSTRAP = 1_000  # resamples for test-set confidence intervals\n",
 "data.py": "\"\"\"Loading, cleaning, splitting and exploratory analysis of the text corpora.\"\"\"\nimport html\nimport re\nfrom collections import Counter\nfrom dataclasses import dataclass, field\n\nimport matplotlib\nimport matplotlib.ticker\n\nmatplotlib.use(\"Agg\")\nimport matplotlib.pyplot as plt\nimport numpy as np\nfrom sklearn.feature_extraction.text import ENGLISH_STOP_WORDS\nfrom sklearn.model_selection import train_test_split\n\nfrom .config import DATASETS, QUICK_CAPS, SEED, VAL_FRACTION\n\nSPLITS = (\"fit\", \"val\", \"test\")\nTOKEN = re.compile(r\"[a-z0-9]+(?:'[a-z]+)?\")\nNEGATION = re.compile(r\"\\b(?:not|no|never|nothing|nobody|none|cannot|nor|without)\\b|n't\\b\")\nBROKEN_ENTITY = re.compile(r\"(?<!&)(#\\d+|quot|amp|lt|gt|nbsp|hellip);\")  # AG News drops the '&'\nSPLIT_CONTRACTION = re.compile(r\"\\s+#39;(?=(?:s|t|re|ve|ll|m|d)\\b)\", re.I)  # \"Google #39;s\" -> \"Google's\"\n\n\ndef clean_text(text: str) -> str:\n    \"\"\"Repair HTML entities, drop markup and line-break backslashes, collapse whitespace.\"\"\"\n    text = SPLIT_CONTRACTION.sub(\"'\", str(text))\n    text = html.unescape(html.unescape(BROKEN_ENTITY.sub(r\"&\\1;\", text)))\n    text = re.sub(r\"<[^>]+>\", \" \", text).replace(\"\\\\\", \" \")\n    return \" \".join(text.split())\n\n\ndef tokenize(text: str) -> list[str]:\n    \"\"\"Lower-cased word tokens; keeps contractions such as \"don't\".\"\"\"\n    return TOKEN.findall(text.lower())\n\n\ndef has_negation(text: str) -> bool:\n    return bool(NEGATION.search(text.lower()))\n\n\n@dataclass\nclass Corpus:\n    name: str\n    text: dict          # split -> list[str]\n    y: dict             # split -> np.ndarray of label ids\n    labels: list        # label id -> name\n    max_len: int\n    quality: dict = field(default_factory=dict)\n\n\ndef _stratified_cap(texts, y, n):\n    if n is None or len(y) <= n:\n        return texts, y\n    keep, _ = train_test_split(np.arange(len(y)), train_size=n, stratify=y, random_state=SEED)\n    keep = np.sort(keep)\n    return [texts[i] for i in keep], y[keep]\n\n\ndef load_corpus(name: str, quick: bool = False) -> Corpus:\n    \"\"\"Download a corpus, clean it, cap it, and build leakage-free fit/val/test splits.\"\"\"\n    from datasets import load_dataset\n\n    spec = DATASETS[name]\n    raw = load_dataset(spec[\"hf_id\"])\n    label_feature = raw[\"train\"].features[\"label\"]\n    labels = list(getattr(label_feature, \"names\", None) or [])\n    if not labels:  # e.g. mteb/banking77 stores names in a separate column\n        labels = [s for _, s in sorted({(int(i), s) for i, s in zip(raw[\"train\"][\"label\"], raw[\"train\"][\"label_text\"])})]\n\n    quality, parts = {}, {}\n    for split in (\"train\", \"test\"):\n        rows = [(clean_text(t), int(y)) for t, y in zip(raw[split][\"text\"], raw[split][\"label\"]) if t is not None]\n        kept = [(t, y) for t, y in rows if t]\n        quality[f\"{split}_missing_or_blank\"] = len(raw[split]) - len(kept)\n        quality[f\"{split}_duplicates\"] = len(kept) - len({t for t, _ in kept})\n        cap = spec[f\"{split}_cap\"]\n        if quick:\n            cap = min(cap or np.inf, QUICK_CAPS[split == \"test\"])\n        parts[split] = _stratified_cap([t for t, _ in kept], np.array([y for _, y in kept]), cap)\n\n    texts, y = parts[\"train\"]\n    fit_idx, val_idx = train_test_split(np.arange(len(y)), test_size=VAL_FRACTION, stratify=y, random_state=SEED)\n\n    # Exact-duplicate leakage: a text seen in test may not be used for fitting or selection.\n    test_set = set(parts[\"test\"][0])\n    val_idx_clean = [i for i in val_idx if texts[i] not in test_set]\n    val_set = {texts[i] for i in val_idx_clean}\n    fit_idx_clean = [i for i in fit_idx if texts[i] not in test_set and texts[i] not in val_set]\n    quality[\"removed_from_fit\"] = len(fit_idx) - len(fit_idx_clean)\n    quality[\"removed_from_val\"] = len(val_idx) - len(val_idx_clean)\n\n    text = {\"fit\": [texts[i] for i in fit_idx_clean], \"val\": [texts[i] for i in val_idx_clean], \"test\": parts[\"test\"][0]}\n    labels_y = {\"fit\": y[fit_idx_clean], \"val\": y[val_idx_clean], \"test\": parts[\"test\"][1]}\n    return Corpus(name, text, labels_y, labels, spec[\"max_len\"], quality)\n\n\ndef run_eda(corpus: Corpus, out_dir):\n    \"\"\"Fit-split statistics and a three-panel figure: class balance, length, frequent words.\"\"\"\n    texts, y = corpus.text[\"fit\"], corpus.y[\"fit\"]\n    tokens = [tokenize(t) for t in texts]\n    lengths = np.array([len(t) for t in tokens])\n    counts = np.bincount(y, minlength=len(corpus.labels))\n    freq = Counter(w for doc in tokens for w in doc)\n    content = Counter({w: n for w, n in freq.items() if w not in ENGLISH_STOP_WORDS})\n    stats = {\n        \"n_fit\": len(texts), \"n_val\": len(corpus.text[\"val\"]), \"n_test\": len(corpus.text[\"test\"]),\n        \"classes\": len(corpus.labels), \"min_class\": int(counts.min()), \"max_class\": int(counts.max()),\n        \"length_mean\": float(lengths.mean()), \"length_median\": float(np.median(lengths)),\n        \"length_p95\": float(np.percentile(lengths, 95)), \"length_max\": int(lengths.max()),\n        \"vocabulary\": len(freq), \"hapax_share\": sum(n == 1 for n in freq.values()) / len(freq),\n        \"negation_share\": float(np.mean([has_negation(t) for t in texts])),\n        \"top_words\": freq.most_common(15), \"top_content_words\": content.most_common(15),\n        **corpus.quality,\n    }\n\n    fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))\n    ax[0].bar(range(len(counts)), np.sort(counts)[::-1], color=\"#4C72B0\")\n    ax[0].set(title=\"Fit-split class sizes (sorted)\", xlabel=\"class rank\", ylabel=\"examples\")\n    ax[0].xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True))\n    ax[1].hist(lengths, bins=40, color=\"#4C72B0\")\n    ax[1].axvline(stats[\"length_p95\"], color=\"#C44E52\", ls=\"--\", label=f\"P95 = {stats['length_p95']:.0f}\")\n    ax[1].set(title=\"Words per text\", xlabel=\"words\", ylabel=\"texts\")\n    ax[1].legend()\n    words, n = zip(*content.most_common(15))\n    ax[2].barh(words[::-1], n[::-1], color=\"#4C72B0\")\n    ax[2].set(title=\"Top content words (stop words removed)\", xlabel=\"count\")\n    fig.tight_layout()\n    fig.savefig(out_dir / \"eda.png\", dpi=150)\n    plt.close(fig)\n    return stats\n",
 "evaluation.py": "\"\"\"Metrics: accuracy, macro P/R/F1, calibration (Brier, ECE), bootstrap intervals, error analysis.\"\"\"\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support\n\nfrom .config import N_BOOTSTRAP, SEED\n\n\ndef brier(y, proba):\n    \"\"\"Multiclass Brier score: mean over texts of sum_k (p_k - 1[y = k])^2. Lower is better.\"\"\"\n    onehot = np.eye(proba.shape[1])[y]\n    return float(np.mean(np.sum((proba - onehot) ** 2, axis=1)))\n\n\ndef ece(y, proba, n_bins=15):\n    \"\"\"Expected calibration error of the top-label confidence over equal-width bins.\"\"\"\n    conf, correct = proba.max(1), proba.argmax(1) == y\n    bins = np.minimum((conf * n_bins).astype(int), n_bins - 1)\n    return float(sum(abs(correct[bins == b].mean() - conf[bins == b].mean()) * np.mean(bins == b)\n                     for b in np.unique(bins)))\n\n\ndef scores(y, pred, proba=None) -> dict:\n    p, r, f, _ = precision_recall_fscore_support(y, pred, average=\"macro\", zero_division=0)\n    out = {\"accuracy\": accuracy_score(y, pred), \"precision_macro\": p, \"recall_macro\": r, \"f1_macro\": f,\n           \"brier\": np.nan, \"ece\": np.nan}\n    if proba is not None:\n        out.update(brier=brier(y, proba), ece=ece(y, proba))\n    return out\n\n\ndef bootstrap_f1(y, preds: dict, n=N_BOOTSTRAP) -> dict:\n    \"\"\"Resample the test set; return the macro-F1 sample for every model (same resamples for all).\"\"\"\n    rng = np.random.default_rng(SEED)\n    draws = {k: [] for k in preds}\n    for _ in range(n):\n        idx = rng.integers(0, len(y), len(y))\n        for k, p in preds.items():\n            draws[k].append(f1_score(y[idx], p[idx], average=\"macro\", zero_division=0))\n    return {k: np.array(v) for k, v in draws.items()}\n\n\ndef per_class_f1(y, preds: dict, labels) -> pd.DataFrame:\n    ids = np.arange(len(labels))\n    table = {k: f1_score(y, p, labels=ids, average=None, zero_division=0) for k, p in preds.items()}\n    return pd.DataFrame(table, index=pd.Index(labels, name=\"class\")).assign(support=np.bincount(y, minlength=len(labels)))\n\n\ndef top_confusions(y, pred, labels, k=10) -> pd.DataFrame:\n    cm = confusion_matrix(y, pred, labels=np.arange(len(labels)))\n    np.fill_diagonal(cm, 0)\n    flat = np.argsort(cm, axis=None)[::-1][:k]\n    rows = [(labels[i], labels[j], cm[i, j]) for i, j in zip(*np.unravel_index(flat, cm.shape)) if cm[i, j]]\n    return pd.DataFrame(rows, columns=[\"true\", \"predicted\", \"count\"])\n",
 "features.py": "\"\"\"Feature extraction and persistence: TF-IDF (.h5) and frozen BGE-small embeddings (.npy).\"\"\"\nimport warnings\n\nimport h5py\nimport numpy as np\nfrom scipy import sparse\nfrom sklearn.feature_extraction.text import TfidfVectorizer\n\nfrom .config import BGE_MODEL, TFIDF_PARAMS\nfrom .data import SPLITS, tokenize\n\n\ndef device():\n    import torch\n\n    if torch.cuda.is_available():\n        return torch.device(\"cuda\")\n    if torch.backends.mps.is_available():\n        return torch.device(\"mps\")\n    return torch.device(\"cpu\")\n\n\n# ---------------------------------------------------------------- TF-IDF\ndef make_vectorizer(tokenizer: str, stop_words: bool, max_ngram: int) -> TfidfVectorizer:\n    \"\"\"tokenizer: 'regex' (word pattern, punctuation dropped) or 'whitespace' (str.split).\"\"\"\n    return TfidfVectorizer(\n        lowercase=True,\n        tokenizer=tokenize if tokenizer == \"regex\" else str.split,\n        token_pattern=None,\n        stop_words=\"english\" if stop_words else None,\n        ngram_range=(1, max_ngram),\n        dtype=np.float32,\n        **TFIDF_PARAMS,\n    )\n\n\ndef fit_tfidf(vectorizer, text: dict) -> dict:\n    \"\"\"Fit the vocabulary and IDF on the fit split only, then transform every split.\"\"\"\n    with warnings.catch_warnings():  # stop-word list vs custom tokenizer notice\n        warnings.simplefilter(\"ignore\", UserWarning)\n        fit = vectorizer.fit_transform(text[\"fit\"])\n    return {\"fit\": fit, **{s: vectorizer.transform(text[s]) for s in (\"val\", \"test\")}}\n\n\ndef save_sparse(path, X: dict, y: dict):\n    with h5py.File(path, \"w\") as f:\n        for s in SPLITS:\n            m = X[s].tocsr()\n            g = f.create_group(s)\n            g.attrs[\"shape\"] = m.shape\n            for key in (\"data\", \"indices\", \"indptr\"):\n                g.create_dataset(key, data=getattr(m, key), compression=\"gzip\")\n            g.create_dataset(\"y\", data=y[s])\n\n\ndef load_sparse(path) -> dict:\n    with h5py.File(path, \"r\") as f:\n        return {s: sparse.csr_matrix((f[s][\"data\"][:], f[s][\"indices\"][:], f[s][\"indptr\"][:]),\n                                     shape=tuple(f[s].attrs[\"shape\"])) for s in SPLITS}\n\n\n# ---------------------------------------------------------------- BGE-small\ndef encode_bge(text: dict, max_len: int, padding: str = \"dynamic\", batch_size: int = 128) -> dict:\n    \"\"\"Frozen encoder: [CLS] vector of the last layer, L2-normalised (BGE's recommended pooling).\n\n    padding='dynamic' pads each batch to its longest text; 'fixed' pads to max_len.\n    The attention mask hides padding either way, so both give the same embeddings.\n    \"\"\"\n    import torch\n    from transformers import AutoModel, AutoTokenizer\n\n    dev = device()\n    tok = AutoTokenizer.from_pretrained(BGE_MODEL)\n    model = AutoModel.from_pretrained(BGE_MODEL).to(dev).eval()\n    out = {}\n    with torch.inference_mode():\n        for s in SPLITS:\n            chunks = []\n            for i in range(0, len(text[s]), batch_size):\n                batch = tok(text[s][i:i + batch_size], truncation=True, max_length=max_len, return_tensors=\"pt\",\n                            padding=\"longest\" if padding == \"dynamic\" else \"max_length\").to(dev)\n                cls = model(**batch).last_hidden_state[:, 0]\n                chunks.append(torch.nn.functional.normalize(cls, dim=1).cpu().numpy())\n            out[s] = np.concatenate(chunks).astype(np.float32)\n    return out\n\n\ndef save_dense(feature_dir, X: dict):\n    for s in SPLITS:\n        np.save(feature_dir / f\"bge_{s}.npy\", X[s])\n\n\ndef load_dense(feature_dir) -> dict:\n    return {s: np.load(feature_dir / f\"bge_{s}.npy\") for s in SPLITS}\n",
 "models.py": "\"\"\"Classifiers: classical heads for stored features and the end-to-end TextCNN.\"\"\"\nimport random\nfrom collections import Counter\n\nimport numpy as np\nimport torch\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.naive_bayes import MultinomialNB\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.svm import LinearSVC\nfrom torch import nn\n\nfrom .config import SEED, TEXTCNN\nfrom .data import tokenize\nfrom .evaluation import scores\n\n\ndef seed_everything(seed: int = SEED):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n\n\ndef make_classifier(name: str, param: float, scale: bool = False):\n    \"\"\"name in {'nb', 'logreg', 'svm'}; param is NB alpha or the C of LogReg/SVM.\"\"\"\n    if name == \"nb\":\n        model = MultinomialNB(alpha=param)\n    elif name == \"logreg\":\n        model = LogisticRegression(C=param, max_iter=2000, random_state=SEED)\n    else:\n        model = LinearSVC(C=param, max_iter=5000, random_state=SEED)\n    return make_pipeline(StandardScaler(), model) if scale else model\n\n\ndef predict_proba(model, X, n_classes):\n    \"\"\"Class probabilities aligned to all label ids, or None for LinearSVC (no probabilities).\"\"\"\n    if not hasattr(model, \"predict_proba\"):\n        return None\n    proba = np.zeros((X.shape[0], n_classes))\n    proba[:, model.classes_] = model.predict_proba(X)\n    return proba\n\n\n# ---------------------------------------------------------------- TextCNN\nclass TextCNN(nn.Module):\n    \"\"\"Kim (2014): embeddings -> parallel 1-D convolutions -> max-over-time pooling -> linear.\"\"\"\n\n    def __init__(self, vocab_size, n_classes, emb_dim, n_filters, widths, dropout):\n        super().__init__()\n        self.embed = nn.Embedding(vocab_size, emb_dim, padding_idx=0)\n        self.convs = nn.ModuleList(nn.Conv1d(emb_dim, n_filters, w) for w in widths)\n        self.dropout = nn.Dropout(dropout)\n        self.out = nn.Linear(n_filters * len(widths), n_classes)\n\n    def forward(self, ids):\n        x = self.embed(ids).transpose(1, 2)                      # (batch, emb_dim, seq_len)\n        pooled = [torch.relu(conv(x)).amax(dim=2) for conv in self.convs]\n        return self.out(self.dropout(torch.cat(pooled, dim=1)))\n\n\ndef build_vocab(texts):\n    \"\"\"Ids 0 = padding, 1 = unknown; keep frequent fit-split words only.\"\"\"\n    counts = Counter(w for t in texts for w in tokenize(t))\n    words = [w for w, n in counts.most_common(TEXTCNN[\"vocab_size\"] - 2) if n >= TEXTCNN[\"min_count\"]]\n    return {w: i + 2 for i, w in enumerate(words)}\n\n\ndef encode_ids(texts, vocab, max_len):\n    \"\"\"Truncate and right-pad every text to max_len token ids (fixed padding).\"\"\"\n    ids = np.zeros((len(texts), max_len), dtype=np.int64)\n    for row, text in enumerate(texts):\n        seq = [vocab.get(w, 1) for w in tokenize(text)][:max_len] or [1]\n        ids[row, :len(seq)] = seq\n    return torch.from_numpy(ids)\n\n\ndef train_textcnn(corpus, batch_size=32, max_epochs=None, dev=None):\n    \"\"\"Adam + cross-entropy; keep the epoch with the best validation macro-F1 (early stopping).\"\"\"\n    seed_everything()\n    cfg = TEXTCNN\n    max_epochs = max_epochs or cfg[\"max_epochs\"]\n    dev = dev or torch.device(\"cpu\")\n    vocab = build_vocab(corpus.text[\"fit\"])\n    X = {s: encode_ids(corpus.text[s], vocab, corpus.max_len) for s in corpus.text}\n    y = {s: torch.from_numpy(corpus.y[s]) for s in corpus.y}\n    model = TextCNN(len(vocab) + 2, len(corpus.labels), cfg[\"emb_dim\"], cfg[\"n_filters\"],\n                    cfg[\"widths\"], cfg[\"dropout\"]).to(dev)\n    optimiser = torch.optim.Adam(model.parameters(), lr=cfg[\"lr\"])\n    loss_fn = nn.CrossEntropyLoss()\n    generator = torch.Generator().manual_seed(SEED)\n\n    def predict(split):\n        model.eval()\n        with torch.no_grad():\n            logits = torch.cat([model(X[split][i:i + 256].to(dev)).cpu() for i in range(0, len(X[split]), 256)])\n        proba = torch.softmax(logits, dim=1).numpy()\n        return proba.argmax(1), proba\n\n    history, best_f1, best_state, stale = [], -1.0, None, 0\n    for epoch in range(1, max_epochs + 1):\n        model.train()\n        total = 0.0\n        for idx in torch.randperm(len(y[\"fit\"]), generator=generator).split(batch_size):\n            xb, yb = X[\"fit\"][idx].to(dev), y[\"fit\"][idx].to(dev)\n            optimiser.zero_grad()\n            loss = loss_fn(model(xb), yb)\n            loss.backward()\n            optimiser.step()\n            total += loss.item() * len(idx)\n        pred, proba = predict(\"val\")\n        history.append({\"epoch\": epoch, \"train_loss\": total / len(y[\"fit\"]), **scores(corpus.y[\"val\"], pred, proba)})\n        if history[-1][\"f1_macro\"] > best_f1:\n            best_f1, stale = history[-1][\"f1_macro\"], 0\n            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}\n        else:\n            stale += 1\n            if stale >= cfg[\"patience\"]:\n                break\n    model.load_state_dict(best_state)\n    pred, proba = predict(\"test\")\n    best_epoch = max(history, key=lambda h: h[\"f1_macro\"])[\"epoch\"]\n    return pred, proba, history, {\"best_epoch\": best_epoch, \"vocab_size\": len(vocab)}\n",
 "pipeline.py": "\"\"\"End-to-end experiment for one corpus.\n\nOrder: load + clean + split -> EDA -> TF-IDF (traditional) -> BGE-small (modern features)\n-> TextCNN (deep learning) -> metrics and error analysis.\nConfigurations are chosen on validation macro-F1; each chosen model is scored once on test.\n\"\"\"\nimport itertools\nimport json\nimport platform\nimport time\nfrom importlib.metadata import version\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\nfrom .config import BGE_C_GRID, BGE_MODEL, CLASSIFIER_GRID, DATASETS, ROOT, SEED, TEXTCNN, TFIDF_GRID\nfrom .data import has_negation, load_corpus, run_eda\nfrom .evaluation import bootstrap_f1, per_class_f1, scores, top_confusions\nfrom .features import device, encode_bge, fit_tfidf, load_dense, load_sparse, make_vectorizer, save_dense, save_sparse\nfrom .models import make_classifier, predict_proba, seed_everything, train_textcnn\n\nFAMILIES = (\"tfidf\", \"bge\", \"textcnn\")\n\n\ndef _fit_and_score(name, param, X, corpus, scale=False):\n    model = make_classifier(name, param, scale).fit(X[\"fit\"], corpus.y[\"fit\"])\n    proba = predict_proba(model, X[\"val\"], len(corpus.labels))\n    return model, scores(corpus.y[\"val\"], model.predict(X[\"val\"]), proba)\n\n\ndef select_tfidf(corpus, classifiers):\n    \"\"\"Grid over tokenizer x stop words x n-grams x classifier x hyper-parameter (validation only).\"\"\"\n    rows = []\n    for tok, stop, ngram in itertools.product(*TFIDF_GRID.values()):\n        X = fit_tfidf(make_vectorizer(tok, stop, ngram), corpus.text)\n        for clf in classifiers:\n            for param in CLASSIFIER_GRID[clf]:\n                _, val = _fit_and_score(clf, param, X, corpus)\n                rows.append({\"family\": \"TF-IDF\", \"tokenizer\": tok, \"stop_words\": stop, \"max_ngram\": ngram,\n                             \"model\": clf, \"param\": param, **val})\n    return rows, max(rows, key=lambda r: r[\"f1_macro\"])\n\n\ndef run_tfidf(corpus, feature_dir, classifiers):\n    rows, best = select_tfidf(corpus, classifiers)\n    vectorizer = make_vectorizer(best[\"tokenizer\"], best[\"stop_words\"], best[\"max_ngram\"])\n    save_sparse(feature_dir / \"tfidf.h5\", fit_tfidf(vectorizer, corpus.text), corpus.y)\n    X = load_sparse(feature_dir / \"tfidf.h5\")  # the classifier reads the stored features\n    model = make_classifier(best[\"model\"], best[\"param\"]).fit(X[\"fit\"], corpus.y[\"fit\"])\n    config = {k: best[k] for k in (\"tokenizer\", \"stop_words\", \"max_ngram\", \"model\", \"param\")}\n    config[\"vocabulary\"] = len(vectorizer.vocabulary_)\n    return rows, config, model.predict(X[\"test\"]), predict_proba(model, X[\"test\"], len(corpus.labels))\n\n\ndef run_bge(corpus, feature_dir, padding):\n    save_dense(feature_dir, encode_bge(corpus.text, corpus.max_len, padding))\n    X = load_dense(feature_dir)\n    rows, best = [], None\n    for c in BGE_C_GRID:\n        model, val = _fit_and_score(\"logreg\", c, X, corpus, scale=True)\n        rows.append({\"family\": \"BGE-small\", \"model\": \"logreg\", \"param\": c, **val})\n        if best is None or val[\"f1_macro\"] > best[1][\"f1_macro\"]:\n            best = (model, val, c)\n    model, _, c = best\n    config = {\"model\": \"logreg\", \"param\": c, \"dim\": int(X[\"fit\"].shape[1]), \"padding\": padding}\n    return rows, config, model.predict(X[\"test\"]), predict_proba(model, X[\"test\"], len(corpus.labels))\n\n\ndef run_dataset(name, out_root=ROOT, quick=False, families=FAMILIES, classifiers=(\"nb\", \"logreg\", \"svm\"),\n                padding=\"dynamic\", batch_size=32):\n    seed_everything()\n    results_dir = out_root / \"reports\" / \"results\" / name\n    feature_dir = out_root / \"features\" / name\n    results_dir.mkdir(parents=True, exist_ok=True)\n    feature_dir.mkdir(parents=True, exist_ok=True)\n\n    start = time.time()\n\n    def log(msg):\n        print(f\"[{name} {time.time() - start:6.0f}s] {msg}\", flush=True)\n\n    corpus = load_corpus(name, quick)\n    log(\"fit/val/test = \" + \"/\".join(str(len(corpus.y[s])) for s in corpus.y))\n    eda = run_eda(corpus, results_dir)\n    for s in corpus.y:\n        np.save(feature_dir / f\"y_{s}.npy\", corpus.y[s])\n\n    validation, configs, preds, probas = [], {}, {}, {}\n    if \"tfidf\" in families:\n        log(\"TF-IDF grid search\")\n        rows, configs[\"TF-IDF\"], preds[\"TF-IDF\"], probas[\"TF-IDF\"] = run_tfidf(corpus, feature_dir, classifiers)\n        validation += rows\n    if \"bge\" in families:\n        log(\"BGE-small embeddings\")\n        rows, configs[\"BGE-small\"], preds[\"BGE-small\"], probas[\"BGE-small\"] = run_bge(corpus, feature_dir, padding)\n        validation += rows\n    if \"textcnn\" in families:\n        log(\"TextCNN training\")\n        dev = torch.device(\"cuda\") if torch.cuda.is_available() else torch.device(\"cpu\")\n        pred, proba, history, info = train_textcnn(corpus, batch_size, 2 if quick else None, dev)\n        preds[\"TextCNN\"], probas[\"TextCNN\"], configs[\"TextCNN\"] = pred, proba, info\n        pd.DataFrame(history).to_csv(results_dir / \"textcnn_history.csv\", index=False)\n\n    log(\"evaluation\")\n    write_results(corpus, results_dir, validation, configs, preds, probas)\n    meta = {\"dataset\": name, \"source\": DATASETS[name][\"hf_id\"], \"labels\": corpus.labels, \"seed\": SEED, \"quick\": quick,\n            \"max_len\": corpus.max_len, \"batch_size\": batch_size, \"bge_model\": BGE_MODEL, \"textcnn\": TEXTCNN,\n            \"bge_device\": str(device()), \"environment\": _environment()}\n    (results_dir / \"run.json\").write_text(json.dumps(meta, indent=2))\n    (results_dir / \"eda.json\").write_text(json.dumps(eda, indent=2))\n    return pd.read_csv(results_dir / \"test_results.csv\")\n\n\ndef write_results(corpus, out, validation, configs, preds, probas):\n    y, labels, texts = corpus.y[\"test\"], corpus.labels, corpus.text[\"test\"]\n    pd.DataFrame(validation).to_csv(out / \"validation.csv\", index=False)\n\n    draws = bootstrap_f1(y, preds)\n    rows = [{\"family\": f, \"config\": json.dumps(configs[f]), **scores(y, preds[f], probas[f]),\n             \"f1_ci_low\": np.percentile(draws[f], 2.5), \"f1_ci_high\": np.percentile(draws[f], 97.5)}\n            for f in preds]\n    results = pd.DataFrame(rows)\n    results.to_csv(out / \"test_results.csv\", index=False)\n    print(results[[\"family\", \"accuracy\", \"f1_macro\", \"f1_ci_low\", \"f1_ci_high\"]].round(4).to_string(index=False))\n\n    # Paired bootstrap: how often the best family beats each other family on the same resample.\n    best = results.loc[results.f1_macro.idxmax(), \"family\"]\n    pd.DataFrame([{\"best\": best, \"other\": f, \"mean_gap\": (draws[best] - draws[f]).mean(),\n                   \"gap_ci_low\": np.percentile(draws[best] - draws[f], 2.5),\n                   \"gap_ci_high\": np.percentile(draws[best] - draws[f], 97.5),\n                   \"p_best_wins\": (draws[best] > draws[f]).mean()} for f in preds if f != best]\n                 ).to_csv(out / \"paired_bootstrap.csv\", index=False)\n\n    negated = np.array([has_negation(t) for t in texts])\n    pd.DataFrame([{\"family\": f, \"n_negation\": int(negated.sum()), \"n_other\": int((~negated).sum()),\n                   \"acc_negation\": (p == y)[negated].mean(), \"acc_other\": (p == y)[~negated].mean()}\n                  for f, p in preds.items()]).assign(gap=lambda d: d.acc_negation - d.acc_other\n                                                     ).to_csv(out / \"negation.csv\", index=False)\n\n    per_class_f1(y, preds, labels).to_csv(out / \"per_class_f1.csv\")\n    pd.concat([top_confusions(y, p, labels).assign(family=f) for f, p in preds.items()]\n              ).to_csv(out / \"top_confusions.csv\", index=False)\n    pd.DataFrame({\"text\": texts, \"true\": [labels[i] for i in y],\n                  **{f: [labels[i] for i in p] for f, p in preds.items()}}).to_csv(out / \"predictions.csv\", index=False)\n\n\ndef _environment():\n    packages = (\"numpy\", \"pandas\", \"scikit-learn\", \"scipy\", \"torch\", \"transformers\", \"datasets\", \"h5py\")\n    return {\"python\": platform.python_version(), **{p: version(p) for p in packages}}\n"
}
Path("modules").mkdir(exist_ok=True)
for name, code in SOURCES.items():
    (Path("modules") / name).write_text(code)
print("Wrote", ", ".join(SOURCES))

## 3. Configure
`QUICK = True` runs a two-minute smoke test on small subsets (results go to `.smoke/` and are **not** the reported results).
Other options: `FAMILIES` ⊆ {tfidf, bge, textcnn}, `CLASSIFIERS` ⊆ {nb, logreg, svm}, `PADDING` ∈ {dynamic, fixed}.
Grids (tokenizer, stop words, n-grams, C, alpha, TextCNN settings) live in `modules/config.py`.

In [ ]:
QUICK = False
DATASETS = ["ag_news", "rotten_tomatoes", "banking77"]
FAMILIES = ["tfidf", "bge", "textcnn"]
CLASSIFIERS = ["nb", "logreg", "svm"]
PADDING = "dynamic"

## 4. Run EDA, feature extraction, model selection and evaluation

In [ ]:
import pandas as pd, torch
from modules import run_dataset
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")
OUT = Path(".smoke" if QUICK else ".").resolve()
summary = pd.concat([run_dataset(d, OUT, QUICK, FAMILIES, CLASSIFIERS, PADDING).assign(dataset=d) for d in DATASETS])
summary.to_csv(OUT / "reports/results/summary.csv", index=False)

## 5. Results
Held-out test scores (macro-F1 with a 95% bootstrap interval), the EDA figure and the five best validation configurations per corpus.

In [ ]:
from IPython.display import Image, display
cols = ["dataset", "family", "accuracy", "f1_macro", "f1_ci_low", "f1_ci_high", "brier", "ece"]
display(summary[cols].round(4))
for d in DATASETS:
    res = OUT / "reports/results" / d
    print("\n" + d)
    display(Image(str(res / "eda.png")))
    display(pd.read_csv(res / "validation.csv").nlargest(5, "f1_macro").round(4))

## 6. Output files
* `features/<dataset>/` — `tfidf.h5` (sparse TF-IDF, fit/val/test), `bge_{fit,val,test}.npy` (384-d embeddings), `y_*.npy`.
* `reports/results/<dataset>/` — `eda.png|json`, `validation.csv` (all tested configurations), `test_results.csv`,
  `paired_bootstrap.csv`, `per_class_f1.csv`, `top_confusions.csv`, `negation.csv`, `predictions.csv`, `textcnn_history.csv`, `run.json`.

Files in a Colab runtime are temporary; download them before disconnecting.